In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ipywidgets import interact, widgets

combined_df = pd.read_csv('vct_2025_clean.csv')
print(f"Loaded {len(combined_df)} rows")

Loaded 183 rows


In [2]:
metrics = ['ACS', 'ADR', 'KAST', 'KPR', 'APR', 'FKPR', 'FDPR', 'HS%']
regions = ['All'] + sorted(combined_df['Region'].unique().tolist())
teams = ['All'] + sorted(combined_df['Team'].unique().tolist())
events = ['All'] + sorted(combined_df['Event'].unique().tolist())

@interact(
    Region=widgets.Dropdown(options=regions, value='All'),
    Team=widgets.Dropdown(options=teams, value='All'),
    Event=widgets.Dropdown(options=events, value='All'),
    Metric=widgets.Dropdown(options=metrics, value='ACS'),
    Sort_By=widgets.Dropdown(options=['Ascending', 'Descending'], value='Descending')
)
def explore(Region, Team, Event, Metric, Sort_By):
    df = combined_df.copy()
    
    if Region != 'All':
        df = df[df['Region'] == Region]
    if Team != 'All':
        df = df[df['Team'] == Team]
    if Event != 'All':
        df = df[df['Event'] == Event]
    
    ascending = Sort_By == 'Ascending'
    df = df.sort_values(Metric, ascending=ascending)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Bar chart of top 15 players
    top15 = df.head(15)
    colors = {'Americas': '#2ecc71', 'EMEA': '#e74c3c', 'Pacific': '#3498db', 'CN': '#f39c12'}
    bar_colors = [colors.get(r, 'grey') for r in top15['Region']]
    axes[0].barh(top15['Player'] + ' (' + top15['Team'] + ')', top15[Metric], color=bar_colors)
    axes[0].set_title(f'Top 15 Players by {Metric}', fontweight='bold')
    axes[0].set_xlabel(Metric)
    axes[0].invert_yaxis()
    
    # Box plot by region
    sns.boxplot(data=df, x='Region', y=Metric, palette='Set2', ax=axes[1])
    axes[1].set_title(f'{Metric} by Region', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # Show table
    print(df[['Player', 'Team', 'Region', 'Event', Metric]].head(20).to_string(index=False))

interactive(children=(Dropdown(description='Region', options=('All', 'Americas', 'CN', 'EMEA', 'Pacific'), val…

In [3]:
@interact(
    X_Metric=widgets.Dropdown(options=metrics, value='ACS'),
    Y_Metric=widgets.Dropdown(options=metrics, value='ADR'),
    Region=widgets.Dropdown(options=regions, value='All'),
    Event=widgets.Dropdown(options=events, value='All')
)
def scatter_plot(X_Metric, Y_Metric, Region, Event):
    df = combined_df.copy()
    
    if Region != 'All':
        df = df[df['Region'] == Region]
    if Event != 'All':
        df = df[df['Event'] == Event]
    
    colors = {'Americas': '#2ecc71', 'EMEA': '#e74c3c', 'Pacific': '#3498db', 'CN': '#f39c12'}
    
    fig, ax = plt.subplots(figsize=(12, 7))
    
    for region, group in df.groupby('Region'):
        ax.scatter(group[X_Metric], group[Y_Metric], 
                  label=region, color=colors.get(region, 'grey'),
                  alpha=0.7, s=80, edgecolors='black', linewidth=0.5)
        
        # Add player labels for top 5 per region
        top5 = group.nlargest(5, X_Metric)
        for _, row in top5.iterrows():
            ax.annotate(row['Player'], (row[X_Metric], row[Y_Metric]),
                       fontsize=7, alpha=0.8)
    
    ax.set_xlabel(X_Metric, fontsize=12)
    ax.set_ylabel(Y_Metric, fontsize=12)
    ax.set_title(f'{X_Metric} vs {Y_Metric} by Region', fontsize=14, fontweight='bold')
    ax.legend(title='Region')
    plt.tight_layout()
    plt.show()

interactive(children=(Dropdown(description='X_Metric', options=('ACS', 'ADR', 'KAST', 'KPR', 'APR', 'FKPR', 'F…

In [4]:
%matplotlib inline
@interact(
    Event=widgets.Dropdown(options=events, value='All')
)
def region_summary(Event):
    df = combined_df.copy()
    
    if Event != 'All':
        df = df[df['Event'] == Event]
    
    summary = df.groupby('Region')[['ACS', 'ADR', 'KAST', 'KPR', 'FKPR', 'HS%']].agg(['mean', 'median']).round(2)
    summary.columns = ['_'.join(col) for col in summary.columns]
    
    print(f"\nRegional Summary Statistics ({Event}):\n")
    print(summary.to_string())
    
    fig, axes = plt.subplots(2, 3, figsize=(16, 14))
    plot_metrics = ['ACS', 'ADR', 'KAST', 'KPR', 'FKPR', 'HS%']
    colors = {'Americas': '#2ecc71', 'EMEA': '#e74c3c', 'Pacific': '#3498db', 'CN': '#f39c12'}
    
    for i, metric in enumerate(plot_metrics):
        ax = axes[i//3][i%3]
        means = df.groupby('Region')[metric].mean()
        bars = ax.bar(means.index, means.values, 
                     color=[colors.get(r, 'grey') for r in means.index],
                     edgecolor='black')
        ax.set_title(f'{metric}', fontweight='bold')
        ax.set_ylabel(metric)
        for bar, val in zip(bars, means.values):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                   f'{val:.1f}', ha='center', va='bottom', fontsize=9)
    
    plt.suptitle(f'Regional Performance Summary — {Event}', fontsize=14, fontweight='bold')
    plt.subplots_adjust(hspace=0.4, wspace=0.3)
    plt.show()

interactive(children=(Dropdown(description='Event', options=('All', 'Champions 2025', 'Masters Bangkok 2025', …

In [5]:
@interact(
    Metric=widgets.Dropdown(options=metrics, value='ACS'),
    Region=widgets.Dropdown(options=regions, value='All')
)
def per_event_breakdown(Metric, Region):
    df = combined_df.copy()
    
    if Region != 'All':
        df = df[df['Region'] == Region]
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Bar chart - mean per region per event
    event_region = df.groupby(['Event', 'Region'])[Metric].mean().unstack()
    event_region.plot(kind='bar', ax=axes[0], 
                      color=['#2ecc71', '#f39c12', '#e74c3c', '#3498db'],
                      edgecolor='black')
    axes[0].set_title(f'Average {Metric} by Event and Region', fontweight='bold')
    axes[0].set_xlabel('Event')
    axes[0].set_ylabel(Metric)
    axes[0].tick_params(axis='x', rotation=30)
    axes[0].legend(title='Region')
    
    # Box plot per event
    sns.boxplot(data=df, x='Event', y=Metric, hue='Region', 
                palette={'Americas': '#2ecc71', 'CN': '#f39c12', 
                         'EMEA': '#e74c3c', 'Pacific': '#3498db'},
                ax=axes[1])
    axes[1].set_title(f'{Metric} Distribution by Event and Region', fontweight='bold')
    axes[1].tick_params(axis='x', rotation=30)
    axes[1].legend(title='Region', bbox_to_anchor=(1.05, 1))
    
    plt.tight_layout()
    plt.show()

interactive(children=(Dropdown(description='Metric', options=('ACS', 'ADR', 'KAST', 'KPR', 'APR', 'FKPR', 'FDP…

In [6]:
@interact(
    Region=widgets.Dropdown(options=regions, value='All'),
    Event=widgets.Dropdown(options=events, value='All'),
    Metric=widgets.Dropdown(options=metrics, value='ACS'),
    Top_N=widgets.IntSlider(min=5, max=20, step=5, value=10)
)
def top_performers(Region, Event, Metric, Top_N):
    df = combined_df.copy()
    
    if Region != 'All':
        df = df[df['Region'] == Region]
    if Event != 'All':
        df = df[df['Event'] == Event]
    
    top = df.nlargest(Top_N, Metric)
    
    colors_map = {'Americas': '#2ecc71', 'EMEA': '#e74c3c', 'Pacific': '#3498db', 'CN': '#f39c12'}
    bar_colors = [colors_map.get(r, 'grey') for r in top['Region']]
    
    fig, ax = plt.subplots(figsize=(12, 7))
    bars = ax.barh(top['Player'] + ' (' + top['Team'] + ')', top[Metric], 
                   color=bar_colors, edgecolor='black')
    ax.set_title(f'Top {Top_N} Players by {Metric}', fontsize=14, fontweight='bold')
    ax.set_xlabel(Metric)
    ax.invert_yaxis()
    
    # Add value labels
    for bar, val in zip(bars, top[Metric]):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
               f'{val:.1f}', va='center', fontsize=9)
    
    # Legend
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=c, label=r) for r, c in colors_map.items()]
    ax.legend(handles=legend_elements, title='Region')
    
    plt.tight_layout()
    plt.show()
    
    print(top[['Player', 'Team', 'Region', 'Event', Metric]].to_string(index=False))

interactive(children=(Dropdown(description='Region', options=('All', 'Americas', 'CN', 'EMEA', 'Pacific'), val…

In [11]:
@interact(
    Region=widgets.Dropdown(options=['All', 'Americas', 'EMEA', 'Pacific', 'CN'], value='All'),
    Event=widgets.Dropdown(options=['All', 'Masters Bangkok 2025', 'Masters Toronto 2025', 'Champions 2025'], value='All')
)
def agent_analysis(Region, Event):
    df = combined_df.copy()
    
    if Region != 'All':
        df = df[df['Region'] == Region]
    if Event != 'All':
        df = df[df['Event'] == Event]
    
    # Extract individual agents
    agent_list = []
    for _, row in df.iterrows():
        if pd.isna(row['Agents']):
            continue
        agents = str(row['Agents']).split(', ')
        seen = set()
        for agent in agents:
            agent = agent.strip()
            if agent and agent not in seen:
                seen.add(agent)
                agent_list.append({'Agent': agent, 'Region': row['Region'], 'Event': row['Event']})
    
    agent_df = pd.DataFrame(agent_list)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    
    # Overall top agents
    top_agents = agent_df['Agent'].value_counts().head(10)
    top_agents.plot(kind='bar', ax=axes[0], color='#3498db', edgecolor='black')
    axes[0].set_title('Top 10 Most Picked Agents', fontweight='bold')
    axes[0].set_xlabel('Agent')
    axes[0].set_ylabel('Pick Count')
    axes[0].tick_params(axis='x', rotation=45)
    
    # Agent picks by region
    agent_region = agent_df.groupby(['Region', 'Agent']).size().unstack(fill_value=0)
    top8 = agent_df['Agent'].value_counts().head(8).index
    agent_region[top8].plot(kind='bar', ax=axes[1], edgecolor='black')
    axes[1].set_title('Top 8 Agent Picks by Region', fontweight='bold')
    axes[1].set_xlabel('Region')
    axes[1].set_ylabel('Pick Count')
    axes[1].tick_params(axis='x', rotation=0)
    axes[1].legend(title='Agent', bbox_to_anchor=(1.05, 1))
    
    plt.tight_layout()
    plt.show()
    
    print("\nAgent pick counts by region:")
    print(agent_df.groupby(['Region', 'Agent']).size().unstack(fill_value=0)[top8])

interactive(children=(Dropdown(description='Region', options=('All', 'Americas', 'EMEA', 'Pacific', 'CN'), val…

In [8]:
print(combined_df['Agents'].head(20))

0            Viper, Vyse, Omen
1       Viper, Chamber, Cypher
2                   Tejo, Sova
3             Raze, Yoru, Jett
4           Yoru, Fade, Cypher
5                         Tejo
6            Raze, Astra, Omen
7        Brimstone, Fade, Sova
8           Breach, Sova, Fade
9             Raze, Neon, Yoru
10      Brimstone, Omen, Astra
11        Vyse, Viper, Chamber
12          Fade, Breach, Omen
13        Viper, Vyse, Killjoy
14    Viper, Cypher, Brimstone
15                  Yoru, Raze
16      Omen, Brimstone, Astra
17            Jett, Raze, Yoru
18            Fade, Kayo, Sova
19      Omen, Brimstone, Astra
Name: Agents, dtype: str


In [9]:
combined_df = pd.read_csv('vct_2025_clean.csv')
print(f"Loaded {len(combined_df)} rows")
print(combined_df[['Player', 'Team', 'Agents']].head(5))

Loaded 183 rows
    Player Team                  Agents
0  CHICHOO  EDG       Viper, Vyse, Omen
1     nAts   TL  Viper, Chamber, Cypher
2    trent   G2              Tejo, Sova
3    Derke  VIT        Raze, Yoru, Jett
4      iZu   T1      Yoru, Fade, Cypher


In [10]:
from matplotlib.patches import FancyArrowPatch
import numpy as np

@interact(
    Event=widgets.Dropdown(options=['All', 'Masters Bangkok 2025', 'Masters Toronto 2025', 'Champions 2025'], value='All')
)
def radar_chart(Event):
    df = combined_df.copy()
    
    if Event != 'All':
        df = df[df['Event'] == Event]
    
    metrics_radar = ['ACS', 'ADR', 'KAST', 'KPR', 'FKPR', 'HS%']
    regions = ['Americas', 'EMEA', 'Pacific', 'CN']
    colors = {'Americas': '#2ecc71', 'EMEA': '#e74c3c', 'Pacific': '#3498db', 'CN': '#f39c12'}
    
    # Normalise data 0-1
    region_means = df.groupby('Region')[metrics_radar].mean()
    region_norm = (region_means - region_means.min()) / (region_means.max() - region_means.min())
    
    N = len(metrics_radar)
    angles = [n / float(N) * 2 * np.pi for n in range(N)]
    angles += angles[:1]
    
    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    
    for region in regions:
        values = region_norm.loc[region].tolist()
        values += values[:1]
        ax.plot(angles, values, linewidth=2, linestyle='solid', 
                label=region, color=colors[region])
        ax.fill(angles, values, alpha=0.1, color=colors[region])
    
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(metrics_radar, size=12)
    ax.set_title('Regional Playstyle Radar Chart', size=15, fontweight='bold', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
    
    plt.tight_layout()
    plt.show()

interactive(children=(Dropdown(description='Event', options=('All', 'Masters Bangkok 2025', 'Masters Toronto 2…